# Canada baby names - clean and organise files

## Load libraries and data

In [20]:
# load libraries
import pandas as pd
import pickle
import numpy as np
from collections import Counter

In [21]:
# load files
ontario_f = '../data/canada/author_analysis/ontario_baby_names_f.csv'
ontario_m = '../data/canada/author_analysis/ontario_baby_names_m.csv'
quebec_f = '../data/canada/author_analysis/quebec_baby_names_f.csv'
quebec_m = '../data/canada/author_analysis/quebec_baby_names_m.csv'

m_ont = pd.read_csv(ontario_m)
f_ont = pd.read_csv(ontario_f)

m_que = pd.read_csv(quebec_m)
f_que = pd.read_csv(quebec_f)

## Ontario baby names

In [22]:
def ont_names(ont_in):
    ont = ont_in.copy()
    ont.rename(columns = {"Year/Année": 'year', "Name/Nom": 'name', "Frequency/Fréquence": 'freq'}, inplace = True)
    
    ont['name'] = ont['name'].str.strip().str.lower() # making names lowercase
    df = ont[['name', 'freq']]
    
    df1 = df.groupby('name').sum() # number of appearance of name over years
    df2 = df.groupby('name').count() # number of years we see the name have records
    
    df1 = df1.reset_index()
    df2 = df2.reset_index()
    df2 = df2.rename(columns={'freq': 'years'})
    
    merged_df = pd.merge(df1, df2, on='name', how='inner') # combining total appearances with year frequency
    merged_df['avg_freq'] = merged_df['freq']/merged_df['years']
    
    merged_df = merged_df[merged_df['name'].str.len() > 1].reset_index(drop = True) # removing single letter rows e.g. 'a'
    ont_names = merged_df['name'].tolist()

    out_dict = dict(zip(merged_df['name'], merged_df['avg_freq']))

    return merged_df, ont_names, out_dict

In [23]:
f_merged_df, f_ont_names, f_ont_dict = ont_names(f_ont)
m_merged_df, m_ont_names, m_ont_dict = ont_names(m_ont)

## Quebec baby names

In [24]:
def que_names(que_in):
    que = que_in.copy()
    
    que.rename(columns = {"Prénom/Année": 'name'}, inplace = True)
    que = que.dropna(subset=['name']).reset_index(drop = True)
    
    que.replace('< 5', '2', inplace=True) # < 5 = 1-4
    que.iloc[:,1:] = que.iloc[:,1:].astype(int)
    
    que['sum'] = que.iloc[:,1:].sum(axis = 1)
    que['name'] = que['name'].str.strip().str.lower()
    que['non_zero'] = (que.iloc[:,1:-1] != 0).sum(axis=1)

    que['avg_non_zero_freq'] = que['sum']/que['non_zero'] # THIS SHOULD BE DIVIDED BY NUMBER OF NON-ZERO YEARS

    que = que[que['name'].str.len() > 1] # removing single letter rows e.g. 'a'
    que['name'] = [' '.join([i for i in x.split(" ") if len(i) > 1]) for x in que['name']] # removing single letter initials from names e.g 'a catherine'
    que = que[que['name'].str.len() > 1] # dropping any names which are now empty strings due to the above process
    
    # have duplicate names due to the above process e.g. 'a catherine' and 'catherine m' become 'catherine'
    # taking the row with the highest sum and removing all the other rows so no duplicate names
    que = que.sort_values('sum', ascending=False).drop_duplicates('name').reset_index(drop = True) 
    que = que.sort_values(by = 'name').reset_index(drop = True) # sort in alphabetical order
    
    que_names = que['name'].tolist()
    
    out_dict = dict(zip(que['name'], que['avg_non_zero_freq']))

                                
    return que, que_names, out_dict

In [25]:
f2_que, f_que_names, f_que_dict = que_names(f_que)
m2_que, m_que_names, m_que_dict = que_names(m_que)

## Check how many lists each name belongs to (will be 1, 2, 3, or 4)

In [26]:
# make list of all names in all four lists
all_names = f_que_names + m_que_names + f_ont_names + m_ont_names
all_names = list(set(all_names))

print(f"Names in F-QUE: {len(f_que_names)}")
print(f"Names in M-QUE: {len(m_que_names)}")
print(f"Names in F-ONT: {len(f_ont_names)}")
print(f"Names in M-ONT: {len(m_ont_names)}")


# for each name - list which lists it is part of
name_check_d = {}
for n in all_names:
    name_check_d[n] = []
    if n in f_que_names:
        name_check_d[n].append('fque')
    if n in m_que_names:
        name_check_d[n].append('mque')
    if n in f_ont_names:
        name_check_d[n].append('font')
    if n in m_ont_names:
        name_check_d[n].append('mont')
        
# see how many names are in 2-4 lists
name_check_d2 = {k: len(v) for k,v in name_check_d.items()}

print(Counter(name_check_d2.values()))

Names in F-QUE: 41067
Names in M-QUE: 40102
Names in F-ONT: 5107
Names in M-ONT: 4442
Counter({1: 67188, 2: 8338, 3: 1658, 4: 470})


## For unambiguous names, categorise them as 'male' or 'female'

1. A name is in only ONE of the four lists: no change - give it that list's gender
2. A name is in TWO lists
    - if both lists are same gender - give it that gender
    - if both lists are different genders: give it the gender value with the highest frequency, provided that the absolute value of the difference in frequency between both lists is >=10
3. A name is in THREE lists: give it the gender value of the majority of the lists (i.e. 2/3 of the lists)
4. A name is in FOUR lists: for each province, compare the difference in gender frequency values (need abs(diff) >= 10). If both have non-significant differences, consider the name ambigious (and return None). If one has a non-significant difference, use this gender categorisation. If both have significant differences: if gender categorisation are the same for both provinces, return that classification, if they are different then consider the name ambigious and return None.

Names in Step 2 and Step 4 which do not have a difference (or differences) in frequency values greater than or equal to 10 will be considered ambiguous

In [27]:
def three_test(l):
    gen_list = [1 for i in l if i[0] == 'f']
    if sum(gen_list) == 2:
        gen = 'f'
    else:
        gen = 'm'
    return gen

def check_diff(name, d1, d2, d1_tag, d2_tag):
    diff = d1[name] - d2[name]
    if diff >= 10:
        return d1_tag[0]
    elif diff <= -10:
        return d2_tag[0]
    else:
        return None

def two_diff_test(name, l, f_ont_dict, m_ont_dict, f_que_dict, m_que_dict): 
    if 'font' in l and 'mont' in l:
        gen_val = check_diff(name, f_ont_dict, m_ont_dict, 'font', 'mont')
    if 'font' in l and 'mque' in l:
        gen_val = check_diff(name, f_ont_dict, m_que_dict, 'font', 'mque')
    if 'fque' in l and 'mont' in l:
        gen_val = check_diff(name, f_que_dict, m_ont_dict, 'fque', 'mont')
    if 'fque' in l and 'mque' in l:
        gen_val = check_diff(name, f_que_dict, m_que_dict, 'fque', 'mque')

    return gen_val


def four_diff_test(name, f_ont_dict, m_ont_dict, f_que_dict, m_que_dict):
    
    gen_ont = check_diff(name, f_ont_dict, m_ont_dict, 'font', 'mont')
    gen_que = check_diff(name, f_que_dict, m_que_dict, 'fque', 'mque')
    
    if gen_ont == None and gen_que == None: # small differences in gender frequencies
        return None
    elif gen_ont == None and gen_que != None: # one difference is significant, other is not
        return gen_que
    elif gen_ont != None and gen_que == None: # one difference is significant, other is not
        return gen_ont
    elif gen_ont != gen_que: # both differences are significant but in opposite genders, consider name ambiguous
        return None
    elif gen_ont == gen_que: # both differences significant for same gender, return that gender
        return gen_ont

In [28]:
final_dict_test = {}
for x in name_check_d.keys():
    l = name_check_d[x]
    
    if len(l) == 1:
        final_dict_test[x] = l[0][0]
    
    elif len(l) == 2:
        if l[0][0] == l[1][0]:
            final_dict_test[x] = l[0][0]
        else:
            final_dict_test[x] = two_diff_test(x, l, f_ont_dict, m_ont_dict, f_que_dict, m_que_dict)
    
    elif len(l) == 3:
        final_dict_test[x] = three_test(l)
    
    elif len(l) == 4:
        final_dict_test[x] = four_diff_test(x, f_ont_dict, m_ont_dict, f_que_dict, m_que_dict)

In [29]:
ignore = []
for k in final_dict_test.keys():
    if final_dict_test[k] == None:
        ignore.append(k)

print(len(ignore))

2629


In [1]:
#ignore

## Combining names into mutually exclusive male and female lists and final gender dictionary


In [31]:
f_final = [k for k,v in final_dict_test.items() if v == 'f']
m_final = [k for k,v in final_dict_test.items() if v == 'm']

print(len(f_final))
print(len(m_final))

38012
37013


In [32]:
# sanity check that there are no overlapping names in the two final name lists
for name in f_final:
    if name in m_final:
        print(name)

In [33]:
# create final dict:
final_names_dict = {name: 'female' for name in f_final}
for name in m_final:
    final_names_dict[name] = 'male'

In [2]:
# final_names_dict

In [35]:
# can_authors_in = '../data/canada/author_analysis/can_authors_29jan2026.pkl'
# with open(can_authors_in, 'rb') as f:
#     names_can = pickle.load(f, encoding="utf-8")

In [36]:
# c = 0
# for k in names_can.keys():
#     if names_can[k]['first'] not in final_names_dict.keys():
#         print(names_can[k]['first'])
#         c += 1
        
# print(c)

## Write data to files - uncomment if running for first time

In [40]:
# # write to files
# with open('../data/canada/author_analysis/can_baby_names.pkl', 'wb') as f:
#     pickle.dump(final_names_dict, f)
    
# # final female names list
# with open('../data/canada/author_analysis/can_female_baby_names.txt', 'w') as file:
#     for item in f_final:
#         file.write(f"{item}\n")

# # final male names list
# with open('../data/canada/author_analysis/can_male_baby_names.txt', 'w') as file:
#     for item in m_final:
#         file.write(f"{item}\n")